In [4]:
import psycopg2
import json
import os
import glob
from dotenv import load_dotenv

load_dotenv()

conn = psycopg2.connect(
    host="localhost",
    port=5432,
    database="ESG_DB",
    user="postgres",
    password=os.environ["DB_PASSWORD"]
)
cur = conn.cursor()
print("Connected to PostgreSQL!")

Connected to PostgreSQL!


In [5]:
cur.execute("""
    CREATE TABLE IF NOT EXISTS companies (
        id SERIAL PRIMARY KEY,
        name VARCHAR(100) UNIQUE NOT NULL,
        region VARCHAR(10),
        country VARCHAR(50),
        sector VARCHAR(50)
    );
""")

cur.execute("""
    CREATE TABLE IF NOT EXISTS filings (
        id SERIAL PRIMARY KEY,
        company_id INTEGER REFERENCES companies(id),
        filing_type VARCHAR(10),
        filing_date DATE,
        file_path VARCHAR(200)
    );
""")

cur.execute("""
    CREATE TABLE IF NOT EXISTS chunks (
        id SERIAL PRIMARY KEY,
        filing_id INTEGER REFERENCES filings(id),
        chunk_index INTEGER,
        text TEXT,
        word_count INTEGER
    );
""")

cur.execute("""
    CREATE TABLE IF NOT EXISTS esg_scores (
        id SERIAL PRIMARY KEY,
        filing_id INTEGER REFERENCES filings(id),
        pillar VARCHAR(1),
        subcategory VARCHAR(50),
        score INTEGER,
        justification TEXT,
        evidence_quote TEXT,
        model_used VARCHAR(50),
        created_at TIMESTAMP DEFAULT NOW()
    );
""")

conn.commit()
print("All 4 tables created!")

All 4 tables created!


In [6]:
companies = [
    # Banks
    ("JPMorgan",           "US", "United States",  "Bank"),
    ("Goldman",            "US", "United States",  "Bank"),
    ("BankOfAmerica",      "US", "United States",  "Bank"),
    ("WellsFargo",         "US", "United States",  "Bank"),
    ("Citigroup",          "US", "United States",  "Bank"),
    ("MorganStanley",      "US", "United States",  "Bank"),
    ("USBancorp",          "US", "United States",  "Bank"),
    ("PNCFinancial",       "US", "United States",  "Bank"),
    # Capital management
    ("BlackRock",          "US", "United States",  "Capital Management"),
    ("StateStreet",        "US", "United States",  "Capital Management"),
    ("CharlesSchwab",      "US", "United States",  "Capital Management"),
    ("BerkshireHathaway",  "US", "United States",  "Capital Management"),
    ("TRowePrice",         "US", "United States",  "Capital Management"),
    # Insurance
    ("MetLife",            "US", "United States",  "Insurance"),
    ("PrudentialFinancial","US", "United States",  "Insurance"),
    ("Aflac",              "US", "United States",  "Insurance"),
    ("Allstate",           "US", "United States",  "Insurance"),
    ("Travelers",          "US", "United States",  "Insurance"),
    # Fintech
    ("Visa",               "US", "United States",  "Fintech"),
    ("Mastercard",         "US", "United States",  "Fintech"),
    ("PayPal",             "US", "United States",  "Fintech"),
    ("Fiserv",             "US", "United States",  "Fintech"),
    # Credit
    ("AmericanExpress",    "US", "United States",  "Credit"),
    ("CapitalOne",         "US", "United States",  "Credit"),
    ("DiscoverFinancial",  "US", "United States",  "Credit"),
    # Exchange/Infrastructure
    ("ICE",                "US", "United States",  "Exchange"),
    ("Nasdaq",             "US", "United States",  "Exchange"),
    ("CMEGroup",           "US", "United States",  "Exchange"),
]

for company in companies:
    cur.execute("""
        INSERT INTO companies (name, region, country, sector)
        VALUES (%s, %s, %s, %s)
        ON CONFLICT (name) DO NOTHING;
    """, company)

conn.commit()
print(f"Inserted {len(companies)} companies!")

Inserted 28 companies!


In [7]:
chunk_files = glob.glob("data/*_chunks.json")
print(f"Found {len(chunk_files)} chunk files\n")

for file_path in chunk_files:
    with open(file_path, "r", encoding="utf-8") as f:
        data = json.load(f)

    company_name = data["company"]
    filing_date  = data["filingDate"]
    chunks       = data["chunks"]

    # Find company in database
    cur.execute("SELECT id FROM companies WHERE name = %s", (company_name,))
    result = cur.fetchone()
    if not result:
        print(f"  Skipping {company_name} — not found in companies table")
        continue
    company_id = result[0]

    # Check if filing already exists to avoid duplicates
    cur.execute("""
        SELECT id FROM filings 
        WHERE company_id = %s AND filing_date = %s
    """, (company_id, filing_date))
    if cur.fetchone():
        print(f"  Skipping {company_name} — already in database")
        continue

    # Insert filing
    cur.execute("""
        INSERT INTO filings (company_id, filing_type, filing_date, file_path)
        VALUES (%s, %s, %s, %s)
        RETURNING id;
    """, (company_id, "10-K", filing_date, file_path))
    filing_id = cur.fetchone()[0]

    # Insert all chunks
    for i, chunk_text in enumerate(chunks):
        cur.execute("""
            INSERT INTO chunks (filing_id, chunk_index, text, word_count)
            VALUES (%s, %s, %s, %s);
        """, (filing_id, i, chunk_text, len(chunk_text.split())))

    conn.commit()
    print(f"  Inserted {len(chunks)} chunks for {company_name} ({filing_date})")

print("\nDone! All chunks stored in PostgreSQL.")

Found 28 chunk files

  Inserted 132 chunks for Aflac (2026-02-25)
  Inserted 167 chunks for Allstate (2026-02-25)
  Inserted 137 chunks for AmericanExpress (2026-02-06)
  Inserted 180 chunks for BankOfAmerica (2026-02-25)
  Inserted 141 chunks for BerkshireHathaway (2026-03-02)
  Inserted 133 chunks for BlackRock (2024-02-23)
  Inserted 153 chunks for CapitalOne (2026-02-19)
  Inserted 113 chunks for CharlesSchwab (2026-02-25)
  Inserted 204 chunks for Citigroup (2026-02-20)
  Inserted 108 chunks for CMEGroup (2026-02-26)
  Inserted 132 chunks for DiscoverFinancial (2025-02-20)
  Inserted 83 chunks for Fiserv (2026-02-19)
  Inserted 241 chunks for Goldman (2026-02-25)
  Inserted 146 chunks for ICE (2026-02-05)
  Inserted 236 chunks for JPMorgan (2026-02-13)
  Inserted 52 chunks for Mastercard (2021-03-19)
  Inserted 218 chunks for MetLife (2026-02-19)
  Inserted 151 chunks for MorganStanley (2026-02-19)
  Inserted 126 chunks for Nasdaq (2026-02-12)
  Inserted 89 chunks for PayPal (202

In [8]:
cur.execute("SELECT COUNT(*) FROM companies;")
print(f"Companies: {cur.fetchone()[0]}")

cur.execute("SELECT COUNT(*) FROM filings;")
print(f"Filings:   {cur.fetchone()[0]}")

cur.execute("SELECT COUNT(*) FROM chunks;")
print(f"Chunks:    {cur.fetchone()[0]}")

print("\nPer company:")
cur.execute("""
    SELECT c.name, c.sector, f.filing_date, COUNT(ch.id)
    FROM companies c
    JOIN filings f ON f.company_id = c.id
    JOIN chunks ch ON ch.filing_id = f.id
    GROUP BY c.name, c.sector, f.filing_date
    ORDER BY c.sector, c.name;
""")
for row in cur.fetchall():
    print(f"  {row[0]:25} ({row[1]:20}) {row[2]}  →  {row[3]} chunks")

Companies: 28
Filings:   28
Chunks:    3905

Per company:
  BankOfAmerica             (Bank                ) 2026-02-25  →  180 chunks
  Citigroup                 (Bank                ) 2026-02-20  →  204 chunks
  Goldman                   (Bank                ) 2026-02-25  →  241 chunks
  JPMorgan                  (Bank                ) 2026-02-13  →  236 chunks
  MorganStanley             (Bank                ) 2026-02-19  →  151 chunks
  PNCFinancial              (Bank                ) 2026-02-20  →  167 chunks
  USBancorp                 (Bank                ) 2026-02-23  →  43 chunks
  WellsFargo                (Bank                ) 2026-02-24  →  29 chunks
  BerkshireHathaway         (Capital Management  ) 2026-03-02  →  141 chunks
  BlackRock                 (Capital Management  ) 2024-02-23  →  133 chunks
  CharlesSchwab             (Capital Management  ) 2026-02-25  →  113 chunks
  StateStreet               (Capital Management  ) 2026-02-19  →  178 chunks
  TRowePrice        

In [9]:
cur.close()
conn.close()
print("Connection closed!")

Connection closed!
